# Spatial Mapping Analysis

---

## Overview
Spatially mapping calculated indices with water quality data in `upper_keys_wq.geoJSON` for spatial patterns and trends over the study period 2015-2024,  with overlay using the Rasterio library. Preprocessed `clipped_manifest` with each scenes `.nc` is required for proper display of Sentinel-2 imagery.

1. Statistical Analysis of Spectral Indices
  - Index Correlation Matrix
  - Seasonal Index Comparison (Point Plot)

2. Spatial Map Analysis
  - Full Coverage Analysis of Turbidity vs. NDAVI for 2015-2024 in Upper Keys FL
  - Seagrass Dynamics Upper Keys FL for 2015-2024 Animation using NDAVI


---

In [ ]:
# pip install..

## Imports

In [ ]:
import os
import cartopy.crs as ccrs
import cartopy.feature as cf
import pandas as pd
import geopandas as gpd 
import rioxarray as rxr 
import rasterio as rio 
import xarray as xr 
import matplotlib
import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np 
import seaborn as sns

### Load Data

In [ ]:
aoi_shapefile = "seagrass_project/study_area/UpperKeys_FL.shp"
clipped_manifest = pd.read_csv("processed/clipped_manifest.csv", parse_dates=["date"])
clipped_manifest["tile"] = clipped_manifest["scene_id"].str.split("_").str[0]
indices_summary = pd.read_csv("processed/indices_summary.csv", parse_dates=["time"])
wq_gdf = gpd.read_file("seagrass_project/wq_data/upper_keys_wq.geojson")

In [ ]:
wq_gdf.to_crs("EPSG:32617")
print(wq_gdf.crs)
print(wq_gdf.head())

## Statistical Analysis
### Index Correlation Matrix


In [ ]:
idx_cols = ["NDVI", "NDWI", "NDAVI", "SSSII", "NDTI", "DII"]
corr = df_indices[idx_cols].apply(pd.to_numeric, errors='coerce').corr()

# Check if corr is empty
if corr.empty or corr.isnull().all().all():
    print("Warning: Correlation matrix is empty. Check for NaNs or constant values.")

fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)

ax.set_title("Index Correlation Matrix")
plt.tight_layout()
plt.savefig("Index_Correlation_Matrix.jpeg", dpi=300, bbox_inches='tight')
plt.show()

### Seasonal Point Plot Index Comparison
NDVI, NDAVI, and SSSII are all indicators of seagrass dynamics but capture different aspects:
- **NDVI (nir/red):** baseline for unsubmerged vegetation, less visibility of submerged vegetation because red light penetration is weak in water.
- **NDAVI (nir/blue):** Better visibility of submerged vegetation as blue light penetrates water better than red.
- **SSSII (green/red):** Better visible difference of seagrass from sand and spectral similarity of unsubmerged and submerged vegetation.

Divergence between indices yields useful information. For example high NDVI but low NDAVI may indicate unsubmerged vegetation beds rather than submerged. SSSII and NDVI are tide dependent, with high tides, NDVI is usually close to 0 and SSSII high, in low tides is the reverse.

In [ ]:
#  Load data
df_indices = pd.read_csv("processed/indices_summary.csv", parse_dates=["time"])

# Add Year and Season columns
df_indices['Year'] = df_indices['time'].dt.year
df_indices['Season'] = df_indices['time'].dt.month.map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring', 4:'Spring', 5:'Spring',
    6:'Summer', 7:'Summer', 8:'Summer',
    9:'Fall', 10:'Fall', 11:'Fall'
})

# Define the logical order for the y-axis
season_order = ['Winter', 'Spring', 'Summer', 'Fall']

#  Setup 3x3 Grid
fig, axes = plt.subplots(3, 3, figsize=(20, 20), sharex=True, sharey=True)
axes = axes.flatten()

# Get the last 9 years
years = sorted(df_indices['Year'].unique())[-9:]

# Plot each year
for i, year in enumerate(years):
    ax = axes[i]
    year_data = df_indices[df_indices['Year'] == year]
    
    # Updated: replaced join=False with linestyles='none' to fix the UserWarning
    sns.pointplot(data=year_data, y='Season', x='NDVI', order=season_order,
                  ax=ax, color='#1f77b4', markers='o', linestyles='none', label='NDVI')
    
    sns.pointplot(data=year_data, y='Season', x='NDAVI', order=season_order,
                  ax=ax, color='#ff7f0e', markers='s', linestyles='none', label='NDAVI')
    
    sns.pointplot(data=year_data, y='Season', x='SSSII', order=season_order,
                  ax=ax, color='#2ca02c', markers='^', linestyles='none', label='SSSII')

    # Highlight the Wet Season (Summer/Fall) background
    ax.axhspan('Summer', 'Fall', color='orange', alpha=0.07, zorder=0)
    
    ax.set_title(f"Year: {year}", fontweight='bold', fontsize=16)
    ax.set_xlabel("Index Value")
    ax.set_ylabel("") # Only the left-most plots show the Season labels due to sharey=True
    ax.grid(True, axis='x', linestyle=':', alpha=0.6)

    # Legend handling
    if i == 0:
        ax.legend(title="Index", loc='upper left', bbox_to_anchor=(0, 1.2), ncol=3, frameon=False)
    else:
        if ax.get_legend(): ax.get_legend().remove()

# 5. Final Layout
plt.suptitle("Seasonal Index Comparison: Year-over-Year (2016-2024)", fontsize=26, y=0.97)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("seagrass_project/indices_seasonal_comparison.jpg", dpi=300, bbox_inches='tight')
plt.show()

## Spatial Map Analysis

#### Base Map Function

Wraps Cartopy code into a function to make quick basemaps - projection set up is Mercator, draws coastline, maps gridlines, and zooms to extent of Upper Keys, FL. Easy reuse for any plot that needs a basemap by calling make_basemap(ax).

In [ ]:
def make_basemap(ax, title="Upper Keys FL"):
    """"
    Creates a basic Cartopy basemap with coastline of the Upper Keys AOI.
    Raster overlay shows vegetation by itself.
    """
    crs = ccrs.PlateCarree()
    projection = ccrs.Mercator()
    gl = ax.gridlines(crs=crs, draw_labels=True,
                      linewidth=.6, color='grey', alpha=0.5, linestyle='-.')
    gl.xlabel_style = {"size": 7}
    gl.ylabel_style = {"size": 7}
    ax.add_feature(cf.COASTLINE.with_scale("10m"), lw=0.8, zorder=4)

    lon_min = -80.622
    lon_max = -80.136
    lat_min = 24.849
    lat_max = 25.374
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=crs)
    ax.set_title(title)

##### Sort for Complete Coverage Scenes
`scene_id`'s that end in "H" contain complete coverage of the AOI Upper Florida Keys shapefile and therefore will be sorted for using Pandas for cleaner spatial mapping.

In [ ]:
h_scenes = clipped_manifest[clipped_manifest['tile'].str.endswith('RNH')].copy()
h_scenes['date'] = pd.to_datetime(h_scenes['date'])
h_scenes['year'] = h_scenes['date'].dt.year

# 2. Function to check if a scene is actually "Full" 
def get_valid_pixel_count(path):
    try:
        with xr.open_dataset(path) as ds:
            # Calculate NDAVI and count non-NaN values
            ndavi = (ds.nir - ds.blue) / (ds.nir + ds.blue)
            return np.count_nonzero(~np.isnan(ndavi.values))
    except:
        return 0

# Apply the check to find scenes with the most data
print("Checking scenes for spatial completeness...")
h_scenes['pixel_count'] = h_scenes['clipped_path'].apply(get_valid_pixel_count)

### Annual Full Coverage Analysis 
A 9-subplot map figure for each year utilizing the `make_basemap` function for NDAVI spectral index vs. overlayed Turbidity (NTU), water quality sample results, for the analysis of seagrass dynamics in the Upper Keys, over the study period 2015-2025 to better analyze changes over time with any spatial correlation between turbidity and seagrass distributions.

In [ ]:
# Select the "Fullest" scene for each year
# Sorting by year and pixel_count (descending) so the best one is at the top
yearly_full_scenes = h_scenes.sort_values(['year', 'pixel_count'], ascending=[True, False]).groupby('year').head(1).head(9)

# --- PLOTTING BLOCK ---
fig, axes = plt.subplots(3, 3, figsize=(20, 24), subplot_kw={'projection': ccrs.Mercator()})
plt.subplots_adjust(hspace=0.25, wspace=0.1)

im = None 

for i, (idx, row) in enumerate(yearly_full_scenes.iterrows()):
    ax = axes.flat[i]
    scene_date = row['date']
    
    with xr.open_dataset(row["clipped_path"]) as ds:
        ndavi = (ds.nir - ds.blue) / (ds.nir + ds.blue)
        ndavi_4326 = ndavi.rio.reproject("EPSG:4326")
        left, bottom, right, top = ndavi_4326.rio.bounds()
        
        make_basemap(ax, title=f"Year: {scene_date.year}")
        
        im = ax.imshow(ndavi_4326.values, origin="upper",
                       extent=[left, right, bottom, top],
                       transform=ccrs.PlateCarree(),
                       cmap="YlGn", vmin=-0.1, vmax=0.4, alpha=0.8, zorder=2)
        
        # WQ Overlay
        wq_near = wq_gdf[
            (pd.to_datetime(wq_gdf["Date"]) >= scene_date - pd.Timedelta("30D")) &
            (pd.to_datetime(wq_gdf["Date"]) <= scene_date + pd.Timedelta("30D"))
        ].to_crs("EPSG:4326")
        
        if not wq_near.empty:
            sc = ax.scatter(wq_near.geometry.x, wq_near.geometry.y,
                            c=wq_near["Turbidity (NTU)"],
                            cmap="Reds",
                            s=55,
                            edgecolors="white",
                            transform=ccrs.PlateCarree(),
                            zorder=6
                           )

# Finalize and Save
# Add the Horizontal Colorbar for NDAVI (Bottom)
cbar_ndavi_ax = fig.add_axes([0.2, 0.08, 0.6, 0.012])
fig.colorbar(im, cax=cbar_ndavi_ax, orientation='horizontal', label='NDAVI Index (Seagrass/Benthic)')

# Add the Vertical Colorbar for Turbidity (Right Side / Y-Axis)
if sc:
    cbar_turb_ax = fig.add_axes([0.91, 0.25, 0.015, 0.5]) # [left, bottom, width, height]
    fig.colorbar(sc, cax=cbar_turb_ax, orientation='vertical', label='Turbidity (NTU)')
    
plt.suptitle("Optimized Annual Mapping: NDAVI & Turbidity", fontsize=26, y=0.93)
plt.savefig("seagrass_project/Annual_Full_Coverage_Analysis.jpg", dpi=300, format='jpg', bbox_inches='tight')
plt.show()

### Upper Keys Seagrass Dynamics 2015-2024 Animation
Utilizing IPython's `FuncAnimation(fig, update, frames=N interval=600)` through `matplotlib.animation` to use the user function to open each scene and create an animated time-series of NDAVI for visualizing change in turbidity overtime. IPython function `ani.to_jshtml()` converts the animation interactive HTML. Raster scene files are large so with Matplotlib setting the frame limit with `rcParams` to 50 allowed for no dropping of frames out of 88.

In [ ]:
# use h_scenes sorted chronologically to ensure full-coverage scenes
animation_scenes = h_scenes.sort_values("date")

# setup figure
fig = plt.figure(dpi=120, figsize=(10, 8))
ax = plt.axes(projection=ccrs.Mercator())
make_basemap(ax, title="NDAVI Upper Keys FL 2015-2024")
img_artist = [None]

def update(i):
    row   = animation_scenes.iloc[i]

    with xr.open_dataset(row["clipped_path"]) as ds:
        ndavi  = (ds.nir - ds.blue) / (ds.nir + ds.blue)
        r4326 = ndavi.rio.reproject("EPSG:4326")

        # downsample to reduce memory — every 4th pixel
        r_down = r4326.values[::4, ::4]
        left, bottom, right, top = r4326.rio.bounds()

        if img_artist[0] is not None:
            try:
                img_artist[0].remove()
            except ValueError:
                pass

        img_artist[0] = ax.imshow(
            r_down,
            origin="upper",
            extent=[left, right, bottom, top],
            transform=ccrs.PlateCarree(),
            cmap="YlOrRd",
            alpha=0.85,
            vmin=-0.3, vmax=0.6,
            zorder=2
        )
        # Update title with date
        ax.set_title(f"NDAVI — {row['date'].date()}", fontsize=14)
        ds.close()
        
# Increase MB limit for the full "H" scenes
matplotlib.rcParams['animation.embed_limit'] = 100  

ani = FuncAnimation(fig, update, frames=len(animation_scenes), interval=400)

# Save as GIF without white space
ani.save("seagrass_project/ndti_animation_2015_2024.gif",
         writer="pillow",
         fps=2,
         dpi=80
        )
print(f"Saved animation with {len(animation_scenes)} number of frames.")

HTML(ani.to_jshtml(fps=2, embed_frames=True))

---

## Summary
**Ouputs:**
- Statistical figures showing total and seasonal index correlation to each other.
- 9-map-subplot of NDAVI vs. Turbidity: for visual analysis of seagrass distributions over the study period with changes in turbidity of the water.
- Animated visual of Seagrass Dynamics in Upper Keys FL for the study period 2015-2024.

**Key Findings:**
-   **Statistical Analysis**: The correlation heatmap of spectral indices validated our model in that NDAVI can be directly related to observable seagrass and not confusion of light refraction/reflection over water due to the strong positive correlation of NDAVI & DII. A strong negative relation between NDAVI & SSSII confirm the expected that NDAVI signal drops in the presence of more suspended sediments as indicated by the SSSII. The seasonal comparison of NDAVI, NDVI, and SSSII also validates the expected in that the overall trend of indices converge during summer when growth and density is at maximum and converge Fall/Winter when environmental stress is high following hurricane season and winter migration of snowbirds.
-	**Spatial Map Analysis**: Confirmed spatial relationship between turbidity and seagrass distribution, with areas of high turbidity seeing less seagrass or NDAVI. With more high turbidity levels being clustered around the Northern Coast where NDAVI has the most variation validated by researched field observations for the Upper Keys.
-	**Animation of Seagrass Dynamics across Upper Keys, Fl, for 2015-2024**: Highlighting the changes in seagrass distribution patterns over time with NDAVI index overlay showing the most fluctuations in NDAVI along the Northern coastal portion of the Upper Keys and a sudden drop of NDAVI at the end of 2017.
-   **2017 Disaster to 2018 recovery**
Observed in the animation, seasonal index comparison of NDVI, NDAVI and SSSII, and the optimized annual mapping of NDAVI and turbidity shows a "disaster-recovery" state of seagrass. With a drop of NDAVI from both signal loss and more sediments post Hurricane Irma. Validated with the seasonal comparison by the major divergence of indices in 2017 (disturbance) and snap-back to normal trends in 2018 (recovery).

## Resources and references
- nilbarde. Ans. (2022, Jan 22). How to save output from cells in jupyter notebook. [Stack Overflow](https://stackoverflow.com/questions/72811926/anybody-know-how-to-save-image-output-from-cells-in-jupyter-notebook)
- Norris, W. (2021, Jun 25). [*Visualizing Satellite Data Using Matplotlib and Cartopy.*](https://towardsdatascience.com/visualizing-satellite-data-using-matplotlib-and-cartopy-8274acb07b84/) Towards Data Science.
- Franko, L. (2022, Jun 28). [*Climate Data Visualization with Python.*](https://medium.com/@lubomirfranko/climate-data-visualisation-with-python-visualise-climate-data-using-cartopy-and-xarray-cf35a60ca8ee) Medium.


